# Phenotype Generation: how many times patient changed to another drug from the same group, phenotype per BNF group

**Phenotype Name**: `<bnf_section_short_name>__switch_frequency`

short names mappings are available in: `../../../data/input/bnf_sections.csv`

**Definition**: To quantify the number of times a patient switches from one drug to a different drug within the same British National Formulary (BNF) section (paragraph). This counts the total number of sequential prescription events where the current drug is replaced by a different substance that belongs to the same BNF section as the preceding drug.

**Outliers:** For each BNF Section's phenotype, the standard deviation (σ) is calculated independently. Maximum value is set as μ+8σ, and any phenotype value exceeding this will be capped to the limit.

In [ ]:
import pyspark
import dxpy
import hail as hl
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sys
import json
from scipy.stats import boxcox

In [ ]:
sys.path.append('../')
from functions.bnf_dictionaries_utils import prepare_substance_to_bnf_section_code_dict, prepare_bnf_section_code_to_short_name_dict
from functions.phenotype_filtration_and_normalization import cap_outliers_mu_plus_n_sigma, pivot_multiple_phenotypes

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)
hl.init(sc=sc, default_reference='GRCh38')

In [ ]:
from datetime import datetime
print(f'Timestamp: {datetime.now()}')
print(f'Instance type: {dxpy.describe(dxpy.JOB_ID)["instanceType"]}')
print(f'Hail version: {hl.version()}')
print(f'Spark version: {spark.version}')

### Configuration and Hail tables loading

In [ ]:
input_database = 'prescriptions_db'
input_prescriptions_tb = 'cleaned_prescriptions_splited_to_therapies_v6.2.0.ht'

output_database = 'prescriptions_db'
output_tb = 'change_to_another_drug_from_bnf_section_phenotypes_v6_2_0.ht'

In [ ]:
input_db_id = dxpy.find_one_data_object(name=input_database, classname='database', project=dxpy.PROJECT_CONTEXT_ID)['id']
ht = hl.read_table(f'dnax://{input_db_id}/{input_prescriptions_tb}')

### Substances to section code dictionary preparation

In [ ]:
substances_section_code_dict = prepare_substance_to_bnf_section_code_dict('../../../data/input/drug_list.csv')

In [ ]:
grouped_ht = ht.group_by(ht.eid, ht.substance, ht.tid).aggregate(
    collected_data = hl.agg.collect(
        hl.struct(
            date_info=ht.date_struct,
            interval=ht.interval
        )
    )
)
grouped_ht = grouped_ht.annotate(
    sorted_collected_data = hl.sorted(
        grouped_ht.collected_data,
        key=lambda s: (s.date_info.year, s.date_info.month, s.date_info.day)
    )
).drop('collected_data')
grouped_ht = grouped_ht.persist()

In [ ]:
ht = grouped_ht.annotate(
    first_date_struct = grouped_ht.sorted_collected_data[0].date_info,
    last_date_struct = grouped_ht.sorted_collected_data[-1].date_info
).drop('sorted_collected_data')
ht = ht.persist()

In [ ]:
substances_section_code_dict_hl = hl.literal(substances_section_code_dict)  

In [ ]:
ht = ht.annotate(
    bnf_section_codes = substances_section_code_dict_hl.get(ht.substance, hl.empty_array('str'))
)
ht = ht.persist()

In [ ]:
ht_exploded = ht.explode(ht.bnf_section_codes)
ht_exploded = ht_exploded.rename({'bnf_section_codes': 'bnf_section_code'})
ht_exploded = ht_exploded.persist()

In [ ]:
code_to_short_name_dict = prepare_bnf_section_code_to_short_name_dict('../../../data/input/bnf_sections.csv')

In [ ]:
code_to_short_name_dict_hl = hl.literal(code_to_short_name_dict)

In [ ]:
ht_exploded = ht_exploded.annotate(
    bnf_section_short_name = code_to_short_name_dict_hl.get(ht_exploded.bnf_section_code, hl.empty_array('str'))
).drop('bnf_section_code')
ht_exploded = ht_exploded.explode(ht_exploded.bnf_section_short_name)
ht_exploded = ht_exploded.persist()

In [ ]:
ht_grouped_by_bnf = ht_exploded.group_by(
    ht_exploded.eid,
    ht_exploded.bnf_section_short_name 
).aggregate(
    collected_data = hl.agg.collect(
        hl.struct(
            substance=ht_exploded.substance,
            tid=ht_exploded.tid,
            first_date_struct=ht_exploded.first_date_struct,     
            last_date_struct=ht_exploded.last_date_struct
        )
    )
)
ht_grouped_by_bnf = ht_grouped_by_bnf.annotate(
    sorted_collected_data = hl.sorted(
        ht_grouped_by_bnf.collected_data, 
        key=lambda s: (s.first_date_struct.year, s.first_date_struct.month, s.first_date_struct.day)
    )
).drop('collected_data')

ht_grouped_by_bnf = ht_grouped_by_bnf.persist()

In [ ]:
def is_earlier_or_equal(date1, date2):
    return (date1.year < date2.year) | \
           ((date1.year == date2.year) & (date1.month < date2.month)) | \
           ((date1.year == date2.year) & (date1.month == date2.month) & (date1.day <= date2.day))


In [ ]:
EMPTY_DATE_STRUCT = hl.struct(
    year=hl.missing(hl.tint32), 
    month=hl.missing(hl.tint32), 
    day=hl.missing(hl.tint32)
)


In [ ]:
ht_with_switches = ht_grouped_by_bnf.annotate(
    scan_results = hl.array_scan(
        lambda acc, current: hl.struct(
            switch_count = hl.if_else(
                (current.substance != acc.last_substance) &
                (~is_earlier_or_equal(current.first_date_struct, acc.last_end_date)) & 
                (hl.is_defined(acc.last_substance)), 
                
                acc.switch_count + 1,
                acc.switch_count
            ),
            last_substance = current.substance,
            last_end_date = current.last_date_struct
        ),
        hl.struct(
            switch_count=0, 
            last_substance=hl.missing(hl.tstr), 
            last_end_date=EMPTY_DATE_STRUCT 
        ),
        ht_grouped_by_bnf.sorted_collected_data
    )
)
ht_with_switches = ht_with_switches.persist()

In [ ]:
ht_final = ht_with_switches.annotate(
    num_of_switches = ht_with_switches.scan_results[-1].switch_count
).drop('scan_results', 'sorted_collected_data')
ht_final = ht_final.persist()

In [ ]:
ht_capped_bnf = cap_outliers_mu_plus_n_sigma(
    ht=ht_final, 
    grouping_col_name='bnf_section_short_name', 
    phenotype_col_name='num_of_switches',
    capped_col_name='capped_num_of_switches'
)

In [ ]:
final_ht = pivot_multiple_phenotypes(
    ht=ht_capped_bnf, 
    key_col_name='eid', 
    pivot_col_name='bnf_section_short_name', 
    phenotype_mappings=[
        ('capped_num_of_switches', '_switch_frequency')
    ]
)

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {output_database} LOCATION 'dnax://'")
output_db_id = dxpy.find_one_data_object(name=output_database, classname='database', project=dxpy.PROJECT_CONTEXT_ID)['id']
output_filtered_prescriptions_url = f'dnax://{output_db_id}/{output_tb}'

%time final_ht.key_by('eid').write(output_filtered_prescriptions_url, overwrite=True)